# EQL, übersetztes SQL und handgeschriebenes SQL — Beispiele im Vergleich

Dieses Notebook zeigt, wie dieselbe Abfrage in drei Fassungen aussieht:

1. **EQL**: so schreibt eine forschende Person die Abfrage.
2. **Generiertes SQL**: das, was der Übersetzer daraus automatisch erzeugt. Es wird beim Ausführen der Zelle live übersetzt.
3. **Handgeschriebenes SQL**: so würde eine Person mit Schemakenntnis die Abfrage von Hand formulieren.

Es wird nichts gemessen und nichts getestet. Es geht nur darum, die Abfragen nebeneinander zu lesen. Zahlen und Metriken stehen in `eql_test.ipynb`, Details zu einzelnen Queries in `eql_demo.ipynb`.

## Setup

In [1]:
import sys
sys.path.insert(0, "/home/jovyan/krrood/test") 

%run ./eql_data_setup.ipynb

from IPython.display import Markdown, display

from krrood.entity_query_language.factories import entity, variable, an, set_of
from krrood.ormatic.eql_interface import eql_to_sql
import eql_metrics as m  # hier nur für die Quelltexte der Beispiele

from test.krrood_test.dataset.semantic_world_like_classes import Body, MoveAction

session = create_demo_session()
build_bodies_for_simple_filter(session)


def zeige(titel, eql, generiertes_sql, handgeschriebenes_sql):
    """Zeigt die drei Fassungen einer Query untereinander, mit Syntaxfärbung."""
    display(Markdown(
        f"### {titel}\n\n"
        f"**EQL**\n```python\n{eql.strip()}\n```\n\n"
        f"**Vom Übersetzer generiertes SQL**\n```sql\n{generiertes_sql.strip()}\n```\n\n"
        f"**Handgeschriebenes SQL**\n```sql\n{handgeschriebenes_sql.strip()}\n```"
    ))

print("Bereit.")

ormatic_interface.py generiert (über einen echten pytest --collect-only-Lauf).
create_demo_session() definiert.
build_bodies_for_simple_filter() definiert.
build_container_handle_world(), build_four_variable_world(), build_container_chain() definiert.
OK: 5 Bodies angelegt.
OK: 3-Variablen-Welt mit 2 Bodies, 1 Connection.
OK: 4-Variablen-Welt mit 4 Bodies, 2 Connections.
OK: Kette mit 4 Containern, 3 Connections.
Bereit.


## 1. Einfacher Filter

**Frage:** Welche Körper (`Body`) sind größer als 5?

Ein einzelner Vergleich auf einer Tabelle. Auffällig ist, dass auch hier die Vererbungskette (`BodyDAO`, `WorldEntityDAO`, `SymbolDAO`) gejoint wird. Das braucht auch die handgeschriebene Fassung, weil ein vollständiges `Body`-Objekt über drei Tabellen verteilt ist.

In [2]:
b = variable(type_=Body, domain=[])
t = eql_to_sql(an(entity(b).where(b.size > 5)), session)

zeige("Einfacher Filter",
      m.EQL_SOURCES["Simple filter"],
      str(t.sql_query),
      m.HANDWRITTEN_SQL["Simple filter"])

### Einfacher Filter

**EQL**
```python
b = variable(type_=Body, domain=[])
an(entity(b).where(b.size > 5))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "BodyDAO".database_id, "WorldEntityDAO".database_id AS database_id_1, "SymbolDAO".database_id AS database_id_2, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id 
WHERE "BodyDAO".size > :size_1
```

**Handgeschriebenes SQL**
```sql
SELECT b.database_id, b.name, b.size
FROM BodyDAO b
JOIN WorldEntityDAO w ON w.database_id = b.database_id
JOIN SymbolDAO s ON s.database_id = w.database_id
WHERE b.size > 5;
```

## 2. Filter auf einzelnen Attributen

**Frage:** Welche Positionen (`robot_x`, `robot_y`) haben `MoveAction`s mit einem `robot_x` über 0,5?

Das ist die Art von Abfrage, wie man sie im Robotik-Alltag stellt: nach einer Eigenschaft einer Aktion filtern und nur bestimmte Werte zurückbekommen, ähnlich wie „alle Aktionen, die mit dem linken Arm ausgeführt wurden". In EQL steht nur die Bedingung am Attribut; Tabellen und Verknüpfungen kommen im SQL hinzu.

In [3]:
action = variable(type_=MoveAction, domain=[])
t = eql_to_sql(an(set_of(action.robot_x, action.robot_y).where(action.robot_x > 0.5)), session)

zeige("Filter auf Attributen",
      """action = variable(type_=MoveAction, domain=[])
an(set_of(action.robot_x, action.robot_y).where(action.robot_x > 0.5))""",
      str(t.sql_query),
      """SELECT m.robot_x, m.robot_y
FROM MoveActionDAO m
WHERE m.robot_x > 0.5;""")

### Filter auf Attributen

**EQL**
```python
action = variable(type_=MoveAction, domain=[])
an(set_of(action.robot_x, action.robot_y).where(action.robot_x > 0.5))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "MoveActionDAO".robot_x, "MoveActionDAO".robot_y 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "MoveActionDAO" ON "MoveActionDAO".database_id = "WorldEntityDAO".database_id 
WHERE "MoveActionDAO".robot_x > :robot_x_1
```

**Handgeschriebenes SQL**
```sql
SELECT m.robot_x, m.robot_y
FROM MoveActionDAO m
WHERE m.robot_x > 0.5;
```

## 3. Transitiver Attributzugriff

**Frage:** Welche Position und welche Greifparameter (`rotate_gripper`, `approach_direction`) gehören zu einer `MoveAction`?

`move.grasp_config.rotate_gripper` liest sich wie ein normaler Attributzugriff, geht aber über zwei Tabellen. In EQL muss der Fremdschlüssel (`grasp_config_id`) nicht bekannt sein. Das handgeschriebene SQL kommt hier mit einem JOIN aus, der Übersetzer bindet zusätzlich die Vererbungskette der Aktion ein.

In [4]:
move = variable(type_=MoveAction, domain=[])
t = eql_to_sql(an(set_of(
    move.robot_x, move.robot_y,
    move.grasp_config.rotate_gripper, move.grasp_config.approach_direction,
)), session)

zeige("Transitiver Attributzugriff",
      m.EQL_SOURCES["Transitive attrs"],
      str(t.sql_query),
      m.HANDWRITTEN_SQL["Transitive attrs"])

### Transitiver Attributzugriff

**EQL**
```python
move = variable(type_=MoveAction, domain=[])
an(set_of(move.robot_x, move.robot_y, move.grasp_config.rotate_gripper, move.grasp_config.approach_direction))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "MoveActionDAO".robot_x, "MoveActionDAO".robot_y, "GraspConfigDAO_1".rotate_gripper, "GraspConfigDAO_1".approach_direction 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "MoveActionDAO" ON "MoveActionDAO".database_id = "WorldEntityDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_1" JOIN "WorldEntityDAO" AS "WorldEntityDAO_1" ON "WorldEntityDAO_1".database_id = "SymbolDAO_1".database_id JOIN "GraspConfigDAO" AS "GraspConfigDAO_1" ON "GraspConfigDAO_1".database_id = "WorldEntityDAO_1".database_id) ON "GraspConfigDAO_1".database_id = "MoveActionDAO"._grasp_config_id
```

**Handgeschriebenes SQL**
```sql
SELECT m.robot_x, m.robot_y, g.rotate_gripper, g.approach_direction
FROM MoveActionDAO m
JOIN GraspConfigDAO g ON g.database_id = m.grasp_config_id;
```

## 4. Mehrere Variablen verbinden (3 Variablen)

**Frage:** Welcher Griff (`Handle`) hängt über welche feste Verbindung (`FixedConnection`) an welchem Behälter (`Container`)?

Jede Bedingung zwischen zwei Variablen (`C == FC.parent`, `H == FC.child`) wird zu einem JOIN. In EQL steht nur, wer mit wem verbunden ist; welche Spalten (`_parent_id`, `_child_id`) dafür nötig sind, steht dort nicht.

In [5]:
world_3 = build_container_handle_world()
C = variable(Container, domain=world_3.bodies)
H = variable(Handle, domain=world_3.bodies)
FC = variable(FixedConnection, domain=world_3.connections)
t = eql_to_sql(an(set_of(C, H, FC).where(C == FC.parent, H == FC.child)), session)

zeige("Mehrere Variablen verbinden (3)",
      m.EQL_SOURCES["Multi-join (3 vars)"],
      str(t.sql_query),
      m.HANDWRITTEN_SQL["Multi-join (3 vars)"])

/tmp/ipykernel_6713/3922224982.py:9: SAWarning: An alias is being generated automatically against joined entity Mapper[FixedConnectionDAO(FixedConnectionDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  str(t.sql_query),
/tmp/ipykernel_6713/3922224982.py:9: SAWarning: An alias is being generated automatically against joined entity Mapper[HandleDAO(HandleDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  str(t.sql_query),


### Mehrere Variablen verbinden (3)

**EQL**
```python
C = variable(Container, domain=world.bodies)
H = variable(Handle, domain=world.bodies)
FC = variable(FixedConnection, domain=world.connections)
an(set_of(C, H, FC).where(C == FC.parent, H == FC.child))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "ContainerDAO".database_id, "BodyDAO".database_id AS database_id_1, "WorldEntityDAO".database_id AS database_id_2, "SymbolDAO".database_id AS database_id_3, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size, "HandleDAO_1".database_id AS database_id_4, "BodyDAO_1".database_id AS database_id_5, "WorldEntityDAO_1".database_id AS database_id_6, "SymbolDAO_1".database_id AS database_id_7, "SymbolDAO_1".polymorphic_type AS polymorphic_type_1, "WorldEntityDAO_1"._world_id AS _world_id_1, "BodyDAO_1".name AS name_1, "BodyDAO_1".size AS size_1, "FixedConnectionDAO_1".database_id AS database_id_8, "ConnectionDAO_1".database_id AS database_id_9, "WorldEntityDAO_2".database_id AS database_id_10, "SymbolDAO_2".database_id AS database_id_11, "SymbolDAO_2".polymorphic_type AS polymorphic_type_2, "WorldEntityDAO_2"._world_id AS _world_id_2, "ConnectionDAO_1"._parent_id, "ConnectionDAO_1"._child_id 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id JOIN "ContainerDAO" ON "ContainerDAO".database_id = "BodyDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_2" JOIN "WorldEntityDAO" AS "WorldEntityDAO_2" ON "WorldEntityDAO_2".database_id = "SymbolDAO_2".database_id JOIN "ConnectionDAO" AS "ConnectionDAO_1" ON "ConnectionDAO_1".database_id = "WorldEntityDAO_2".database_id JOIN "FixedConnectionDAO" AS "FixedConnectionDAO_1" ON "FixedConnectionDAO_1".database_id = "ConnectionDAO_1".database_id) ON "ConnectionDAO_1"._parent_id = "ContainerDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_1" JOIN "WorldEntityDAO" AS "WorldEntityDAO_1" ON "WorldEntityDAO_1".database_id = "SymbolDAO_1".database_id JOIN "BodyDAO" AS "BodyDAO_1" ON "BodyDAO_1".database_id = "WorldEntityDAO_1".database_id JOIN "HandleDAO" AS "HandleDAO_1" ON "HandleDAO_1".database_id = "BodyDAO_1".database_id) ON "ConnectionDAO_1"._child_id = "HandleDAO_1".database_id
```

**Handgeschriebenes SQL**
```sql
SELECT c.database_id, c_b.name, c_b.size,
       h.database_id, h_b.name, h_b.size,
       fc.database_id, fcc._parent_id, fcc._child_id
FROM ContainerDAO c
JOIN BodyDAO c_b ON c_b.database_id = c.database_id
JOIN WorldEntityDAO c_w ON c_w.database_id = c_b.database_id
JOIN SymbolDAO c_s ON c_s.database_id = c_w.database_id
JOIN ConnectionDAO fcc ON fcc._parent_id = c.database_id
JOIN FixedConnectionDAO fc ON fc.database_id = fcc.database_id
JOIN WorldEntityDAO fc_w ON fc_w.database_id = fcc.database_id
JOIN SymbolDAO fc_s ON fc_s.database_id = fc_w.database_id
JOIN HandleDAO h ON h.database_id = fcc._child_id
JOIN BodyDAO h_b ON h_b.database_id = h.database_id
JOIN WorldEntityDAO h_w ON h_w.database_id = h_b.database_id
JOIN SymbolDAO h_s ON h_s.database_id = h_w.database_id;
```

## 5. Mehrere Variablen verbinden (4 Variablen)

**Frage:** Dasselbe wie oben, zusätzlich mit einer verschieblichen Verbindung (`PrismaticConnection`), die denselben Container mit einem weiteren Körper verbindet, etwa eine Schublade an einem Schrank mit Griff.

Die EQL-Fassung wächst nur um eine Variable und eine Bedingung, im SQL kommt eine weitere Vererbungskette hinzu.

In [6]:
world_4 = build_four_variable_world()
C4 = variable(Container, domain=world_4.bodies)
H4 = variable(Handle, domain=world_4.bodies)
FC4 = variable(FixedConnection, domain=world_4.connections)
PC4 = variable(PrismaticConnection, domain=world_4.connections)
t = eql_to_sql(an(set_of(C4, H4, FC4, PC4).where(
    C4 == FC4.parent, H4 == FC4.child, C4 == PC4.child,
)), session)

zeige("Mehrere Variablen verbinden (4)",
      m.EQL_SOURCES["Multi-join (4 vars)"],
      str(t.sql_query),
      m.HANDWRITTEN_SQL["Multi-join (4 vars)"])

/tmp/ipykernel_6713/3764172325.py:12: SAWarning: An alias is being generated automatically against joined entity Mapper[FixedConnectionDAO(FixedConnectionDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  str(t.sql_query),
/tmp/ipykernel_6713/3764172325.py:12: SAWarning: An alias is being generated automatically against joined entity Mapper[HandleDAO(HandleDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  str(t.sql_query),
/tmp/ipykernel_6713/3764172325.py:12: SAWarning: An alias is being generated automatically against joined entity Mapper[PrismaticConnectionDAO(PrismaticConnectionDAO)] du

### Mehrere Variablen verbinden (4)

**EQL**
```python
C = variable(Container, domain=world.bodies)
H = variable(Handle, domain=world.bodies)
FC = variable(FixedConnection, domain=world.connections)
PC = variable(PrismaticConnection, domain=world.connections)
an(set_of(C, H, FC, PC).where(C == FC.parent, H == FC.child, C == PC.child))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "ContainerDAO".database_id, "BodyDAO".database_id AS database_id_1, "WorldEntityDAO".database_id AS database_id_2, "SymbolDAO".database_id AS database_id_3, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size, "HandleDAO_1".database_id AS database_id_4, "BodyDAO_1".database_id AS database_id_5, "WorldEntityDAO_1".database_id AS database_id_6, "SymbolDAO_1".database_id AS database_id_7, "SymbolDAO_1".polymorphic_type AS polymorphic_type_1, "WorldEntityDAO_1"._world_id AS _world_id_1, "BodyDAO_1".name AS name_1, "BodyDAO_1".size AS size_1, "FixedConnectionDAO_1".database_id AS database_id_8, "ConnectionDAO_1".database_id AS database_id_9, "WorldEntityDAO_2".database_id AS database_id_10, "SymbolDAO_2".database_id AS database_id_11, "SymbolDAO_2".polymorphic_type AS polymorphic_type_2, "WorldEntityDAO_2"._world_id AS _world_id_2, "ConnectionDAO_1"._parent_id, "ConnectionDAO_1"._child_id, "PrismaticConnectionDAO_1".database_id AS database_id_12, "ConnectionDAO_2".database_id AS database_id_13, "WorldEntityDAO_3".database_id AS database_id_14, "SymbolDAO_3".database_id AS database_id_15, "SymbolDAO_3".polymorphic_type AS polymorphic_type_3, "WorldEntityDAO_3"._world_id AS _world_id_3, "ConnectionDAO_2"._parent_id AS _parent_id_1, "ConnectionDAO_2"._child_id AS _child_id_1 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id JOIN "ContainerDAO" ON "ContainerDAO".database_id = "BodyDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_2" JOIN "WorldEntityDAO" AS "WorldEntityDAO_2" ON "WorldEntityDAO_2".database_id = "SymbolDAO_2".database_id JOIN "ConnectionDAO" AS "ConnectionDAO_1" ON "ConnectionDAO_1".database_id = "WorldEntityDAO_2".database_id JOIN "FixedConnectionDAO" AS "FixedConnectionDAO_1" ON "FixedConnectionDAO_1".database_id = "ConnectionDAO_1".database_id) ON "ConnectionDAO_1"._parent_id = "ContainerDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_1" JOIN "WorldEntityDAO" AS "WorldEntityDAO_1" ON "WorldEntityDAO_1".database_id = "SymbolDAO_1".database_id JOIN "BodyDAO" AS "BodyDAO_1" ON "BodyDAO_1".database_id = "WorldEntityDAO_1".database_id JOIN "HandleDAO" AS "HandleDAO_1" ON "HandleDAO_1".database_id = "BodyDAO_1".database_id) ON "ConnectionDAO_1"._child_id = "HandleDAO_1".database_id JOIN ("SymbolDAO" AS "SymbolDAO_3" JOIN "WorldEntityDAO" AS "WorldEntityDAO_3" ON "WorldEntityDAO_3".database_id = "SymbolDAO_3".database_id JOIN "ConnectionDAO" AS "ConnectionDAO_2" ON "ConnectionDAO_2".database_id = "WorldEntityDAO_3".database_id JOIN "PrismaticConnectionDAO" AS "PrismaticConnectionDAO_1" ON "PrismaticConnectionDAO_1".database_id = "ConnectionDAO_2".database_id) ON "ConnectionDAO_2"._child_id = "ContainerDAO".database_id
```

**Handgeschriebenes SQL**
```sql
SELECT c.database_id, c_b.name, c_b.size,
       h.database_id, h_b.name, h_b.size,
       fc.database_id, fcc._parent_id, fcc._child_id,
       pc.database_id, pcc._parent_id, pcc._child_id
FROM ContainerDAO c
JOIN BodyDAO c_b ON c_b.database_id = c.database_id
JOIN WorldEntityDAO c_w ON c_w.database_id = c_b.database_id
JOIN SymbolDAO c_s ON c_s.database_id = c_w.database_id
JOIN ConnectionDAO fcc ON fcc._parent_id = c.database_id
JOIN FixedConnectionDAO fc ON fc.database_id = fcc.database_id
JOIN WorldEntityDAO fc_w ON fc_w.database_id = fcc.database_id
JOIN SymbolDAO fc_s ON fc_s.database_id = fc_w.database_id
JOIN HandleDAO h ON h.database_id = fcc._child_id
JOIN BodyDAO h_b ON h_b.database_id = h.database_id
JOIN WorldEntityDAO h_w ON h_w.database_id = h_b.database_id
JOIN SymbolDAO h_s ON h_s.database_id = h_w.database_id
JOIN ConnectionDAO pcc ON pcc._child_id = c.database_id
JOIN PrismaticConnectionDAO pc ON pc.database_id = pcc.database_id
JOIN WorldEntityDAO pc_w ON pc_w.database_id = pcc.database_id
JOIN SymbolDAO pc_s ON pc_s.database_id = pc_w.database_id;
```

## 6. Filtern, sortieren, begrenzen

**Frage:** Welche sind die 10 kleinsten Körper mit einer Größe über 5, aufsteigend nach Größe sortiert?

Filter, Sortierung und Begrenzung bleiben in EQL drei benannte Schritte (`where`, `ordered_by`, `limit`). Im SQL werden daraus drei Klauseln mit jeweils eigener Syntax.

In [7]:
b = variable(type_=Body, domain=[])
t = eql_to_sql(an(entity(b).where(b.size > 5).ordered_by(b.size).limit(10)), session)

zeige("Filtern, sortieren, begrenzen",
      m.EQL_SOURCES["Complex combined"],
      str(t.sql_query),
      m.HANDWRITTEN_SQL["Complex combined"])

### Filtern, sortieren, begrenzen

**EQL**
```python
b = variable(type_=Body, domain=[])
an(entity(b).where(b.size > 5).ordered_by(b.size).limit(10))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "BodyDAO".database_id, "WorldEntityDAO".database_id AS database_id_1, "SymbolDAO".database_id AS database_id_2, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id 
WHERE "BodyDAO".size > :size_1 ORDER BY "BodyDAO".size
 LIMIT :param_1
```

**Handgeschriebenes SQL**
```sql
SELECT b.*, w.*, s.*
FROM BodyDAO b
JOIN WorldEntityDAO w ON w.database_id = b.database_id
JOIN SymbolDAO s ON s.database_id = w.database_id
WHERE b.size > 5
ORDER BY b.size
LIMIT 10;
```

## 7. Das Beispiel aus der Einleitung (PickUpAction)

**Frage:** Welcher Arm, welche Greifer-Rotation und welche Anfahrtsrichtung gehören zu einer `PickUpAction`?

Dieses Beispiel steht am Anfang der Arbeit. Es wird hier nicht live übersetzt, weil die echten pycram-Klassen eine ROS2-Umgebung brauchen (siehe `eql_demo.ipynb`, letzter Abschnitt); die Texte stammen aus `eql_metrics.py` und sind identisch mit denen in der Arbeit. Der Übersetzer bindet die Vererbungskette (`DesignatorDAO`, `ActionDescriptionDAO`) mit ein, obwohl keine der gewählten Spalten dort liegt; die handgeschriebene Fassung lässt diese Tabellen weg und ist hier sogar kürzer als EQL.

In [8]:
zeige("PickUpAction-Beispiel (Einleitung)",
      m.PICKUP_EXAMPLE_EQL,
      m.PICKUP_EXAMPLE_TRANSLATOR_SQL,
      m.PICKUP_EXAMPLE_HANDWRITTEN_SQL)

### PickUpAction-Beispiel (Einleitung)

**EQL**
```python
action = variable(type_=PickUpAction, domain=[])
query = an(set_of(
    action.arm,
    action.grasp_description.rotate_gripper,
    action.grasp_description.approach_direction,
))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "PickUpActionDAO".arm, "GraspDescriptionDAO_1".rotate_gripper, "GraspDescriptionDAO_1".approach_direction
FROM "DesignatorDAO"
JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id
JOIN "PickUpActionDAO" ON "PickUpActionDAO".database_id = "ActionDescriptionDAO".database_id
JOIN "GraspDescriptionDAO" AS "GraspDescriptionDAO_1" ON "GraspDescriptionDAO_1".database_id = "PickUpActionDAO".grasp_description_id
```

**Handgeschriebenes SQL**
```sql
SELECT p.arm, g.rotate_gripper, g.approach_direction
FROM PickUpActionDAO p
JOIN GraspDescriptionDAO g ON g.database_id = p.grasp_description_id;
```

## Eigene Query

Hier kann eine eigene Query ausprobiert werden. Die handgeschriebene Fassung muss selbst geschrieben werden.

In [9]:
meine_variable = variable(type_=MoveAction, domain=[])
meine_query = an(set_of(meine_variable.robot_x).where(meine_variable.robot_x > 1))
t = eql_to_sql(meine_query, session)

zeige("Eigene Query",
      """meine_variable = variable(type_=MoveAction, domain=[])
an(set_of(meine_variable.robot_x).where(meine_variable.robot_x > 1))""",
      str(t.sql_query),
      "-- hier eigenes SQL einfügen")

### Eigene Query

**EQL**
```python
meine_variable = variable(type_=MoveAction, domain=[])
an(set_of(meine_variable.robot_x).where(meine_variable.robot_x > 1))
```

**Vom Übersetzer generiertes SQL**
```sql
SELECT "MoveActionDAO".robot_x 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "MoveActionDAO" ON "MoveActionDAO".database_id = "WorldEntityDAO".database_id 
WHERE "MoveActionDAO".robot_x > :robot_x_1
```

**Handgeschriebenes SQL**
```sql
-- hier eigenes SQL einfügen
```